# Deep Sets model walkthrough

Trace the Python implementation through entity features, the shared encoder, grouped sums, and the value MLP. The Energy-attachment example checks full versus incremental evaluation and saved-parent restoration, followed by a joint-gradient check.


**Colab: choose Runtime → Run all.** Setup clones the public [project repository](https://github.com/le21-j/pokemon-tcg-fpga) and installs dependencies automatically.

In [ ]:
# @title 1. Automatic setup {display-mode: "form"}
# @markdown Run All clones the public project and installs dependencies automatically in Colab. Optional overrides can stay at their defaults.
REPOSITORY_PATH = "" # @param {type:"string"}
MOUNT_GOOGLE_DRIVE = False # @param {type:"boolean"}
INSTALL_LOCALLY = False # @param {type:"boolean"}

"""Standard-library Colab setup for the public Pokémon TCG project repository."""

import os
from pathlib import Path
import subprocess
import sys
import tempfile


PUBLIC_REPOSITORY = "https://github.com/le21-j/pokemon-tcg-fpga.git"


def running_in_colab():
    if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
        return True
    try:
        import google.colab
        return True
    except ImportError:
        return False


def _project_ready(root):
    return all((root / name).is_file() for name in (
        "project/arena.py", "project/sample_sdk.py", "project/agent/model.py",
        "project/rl_baseline/train.py", "project/agent_template/agent.py",
        "project/requirements-colab.txt"))


def setup_project(repository_path="", *, repository_url=PUBLIC_REPOSITORY, repository_ref="main",
                  in_colab=None, workspace_parent=None, install_locally=False):
    in_colab = running_in_colab() if in_colab is None else in_colab
    if repository_path.strip():
        root = Path(repository_path).expanduser().resolve()
        if not _project_ready(root):
            raise ValueError("REPOSITORY_PATH must contain the Pokémon TCG project")
    elif not in_colab:
        root = next((p for p in [Path.cwd(), *Path.cwd().parents] if _project_ready(p)), None)
        if root is None:
            raise ValueError("Run inside the project checkout or set REPOSITORY_PATH")
    else:
        parent = Path(workspace_parent) if workspace_parent is not None else (
            Path("/content") if Path("/content").is_dir() else Path.cwd())
        parent.mkdir(parents=True, exist_ok=True)
        name = "pokemon-tcg-fpga"
        root = parent / name
        retry = 0
        while root.exists() and not _project_ready(root):
            retry += 1
            root = parent / f"{name}-{retry}"
        if not root.exists():
            with tempfile.TemporaryDirectory(prefix="tcg-clone-", dir=parent) as temporary:
                checkout = Path(temporary) / "checkout"
                subprocess.run(["git", "clone", "--depth", "1", "--branch", repository_ref,
                                repository_url, str(checkout)],
                               env=dict(os.environ, GIT_TERMINAL_PROMPT="0"), check=True)
                if not _project_ready(checkout):
                    raise RuntimeError("The downloaded repository is missing project files")
                checkout.rename(root)
            print("Cloned the public Pokémon TCG project")
    if in_colab or install_locally:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                               "-r", str(root / "project/requirements-colab.txt")])
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print("Project ready:", root)
    return root


if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = setup_project(REPOSITORY_PATH, install_locally=INSTALL_LOCALLY)

# --- End automatic project setup ---

import inspect
import pandas as pd
import torch
from IPython.display import display, Code

from project.agent.features import Group, ENTITY_FEATURES, position_from_observation
from project.agent.model import DeepSetsValue, batch_positions
from project.agent.evaluator import IncrementalEvaluator
from project.energy_attachment import example_observations

print("PyTorch:", torch.__version__)
print("Project modules loaded")

## Features and the shared card encoder

`position_from_observation()` converts visible card/Pokémon facts into `Entity` records. Each entity has a stable instance ID, one of four groups, and 16 numeric features. The position also contains 32 global features and 32 history slots.

The current `observed-resources-v0` schema describes HP, Energy/Tool counts, roles, statuses, counts, and selection type. Card-mechanics metadata is a later feature update. History slots currently contain zeros. This first example is a synthetic, API-shaped ordinary Energy attachment.

In [ ]:
before_observation, after_observation, alternate_observation = example_observations()
before = position_from_observation(before_observation)
after = position_from_observation(after_observation)
alternate = position_from_observation(alternate_observation)

display(pd.DataFrame([
    {"instance": entity.instance_id, "group": entity.group.name,
     **dict(zip(ENTITY_FEATURES, entity.features.tolist()))}
    for entity in before.entities
]))

`DeepSetsValue.encoder` is the same two-layer MLP for every entity. It maps 16 input features to 16 learned values, with ReLU after each layer. The model's parameter count stays constant as the number of cards changes.

`batch_positions()` pads examples to the largest entity count in the batch and supplies a mask. The pool operation applies that mask **after encoding**, so padded rows contribute zero even when encoder layers have biases.

In [ ]:
torch.manual_seed(7)
model = DeepSetsValue().eval()
print(model)
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters()))
assert sum(parameter.numel() for parameter in model.parameters()) == 8865

entities, groups, globals_, history, mask = batch_positions([before, after])
display(pd.DataFrame([
    {"tensor": "entities", "shape": tuple(entities.shape)},
    {"tensor": "group IDs", "shape": tuple(groups.shape)},
    {"tensor": "mask", "shape": tuple(mask.shape)},
    {"tensor": "globals", "shape": tuple(globals_.shape)},
    {"tensor": "history", "shape": tuple(history.shape)},
]))

## Grouped sums and the value MLP

`pools()` first encodes all entities and masks padding. `scatter_add()` then adds each encoded vector to its designated group. Own board, opposing board, own hand, and public discards each produce 16 values.

These learned coordinates have no assigned interpretation such as HP or attack strength. Training determines which combinations help predict outcomes.

In [ ]:
display(Code(inspect.getsource(DeepSetsValue.pools), language="python"))
with torch.inference_mode():
    encoded = model.encoder(entities)
    pooled = model.pools(entities, groups, mask)
print("Encoded entities:", tuple(encoded.shape))
print("Pooled summaries:", tuple(pooled.shape))
display(pd.DataFrame([
    {"group": group.name,
     "entities before": sum(entity.group == group for entity in before.entities),
     **{f"channel {i}": pooled[0, int(group), i].item() for i in range(4)}}
    for group in Group
]))

`value_from_pools()` flattens the four summaries to 64 values and concatenates the 32 global and 32 history values. The resulting 128 features pass through a 64-unit ReLU layer and one output neuron.

The value is trained against the observer's final return, with win `+1`, draw `0`, and loss `-1`. The weights in this walkthrough use a fixed random initialization, so the displayed values demonstrate the computation.

In [ ]:
display(Code(inspect.getsource(DeepSetsValue.value_from_pools), language="python"))
with torch.inference_mode():
    full_before = model.evaluate_position(before).item()
    full_after = model.evaluate_position(after).item()
print(f"Full value before attachment: {full_before:.9f}")
print(f"Full value after attachment:  {full_after:.9f}")

## Full versus incremental Energy attachment

Full evaluation encodes every entity in the resulting position. `IncrementalEvaluator.sync()` creates a cache of the root position. `update()` matches entities by instance ID, removes vanished contributions, and re-encodes entities whose features or group changed.

For this attachment, the Energy leaves the hand and the target Pokémon gains another attached Energy. The updater subtracts the cached hand-Energy vector and replaces the Pokémon's old board contribution. It also updates global inputs such as hand count and `energyAttached`.

In [ ]:
evaluator = IncrementalEvaluator(model)
evaluator.sync(before)
evaluator.push()

with torch.inference_mode():
    changes = evaluator.update(after)
    incremental_after = evaluator.value().item()
    full_after_tensor = model.evaluate_position(after)
    torch.testing.assert_close(evaluator.value(), full_after_tensor, rtol=1e-5, atol=1e-6)
    batch = batch_positions([after])
    full_pools = model.pools(batch[0], batch[1], batch[4])[0]
    torch.testing.assert_close(evaluator.pooled_state, full_pools, rtol=1e-5, atol=1e-6)

print("Incremental changes:", changes)
assert changes == {"removed": 1, "encoded": 1, "reused": 3}
display(pd.DataFrame([
    {"method": "full recomputation", "value": full_after},
    {"method": "incremental update", "value": incremental_after},
    {"method": "absolute difference", "value": abs(full_after - incremental_after)},
]))

`push()` saves the evaluator state before a search branch. `pop()` restores cached vectors, group sums, globals, and history. The second branch below attaches to the Benched Pokémon from the restored root.

Full and incremental floating-point sums can differ slightly because addition order changes. We compare them with a tolerance. Restoration copies a saved state, so the restored value should match exactly. The future integer reference will require exact arithmetic agreement.

In [ ]:
evaluator.pop()
assert evaluator.value().item() == full_before
print("Restored root:", evaluator.value().item())
evaluator.push()
with torch.inference_mode():
    alternate_changes = evaluator.update(alternate)
    alternate_value = evaluator.value().item()
    torch.testing.assert_close(evaluator.value(), model.evaluate_position(alternate))
evaluator.pop()
assert evaluator.value().item() == full_before
print("Alternative attachment changes:", alternate_changes)
print("Alternative attachment value:", alternate_value)
print("PASS: both attachment branches restore the same root")

## How both learned components receive gradients

During training, `forward()` computes the full model using the current weights. Backpropagation passes the loss gradient through the value MLP, the grouped sums, and the shared encoder. The inference cache belongs to one fixed set of model weights.

The next cell uses an illustrative target of `+1` to check gradient flow. Actual training targets come from completed matches. It leaves the weights unchanged.

In [ ]:
model.zero_grad(set_to_none=True)
predictions = model(*batch_positions([before, after]))
loss = (predictions - 1.0).square().mean()
loss.backward()
gradients = {
    "encoder first layer": model.encoder[0].weight.grad.norm().item(),
    "value MLP first layer": model.value_head[0].weight.grad.norm().item(),
}
assert all(value > 0 for value in gradients.values())
print("Illustrative loss:", loss.item())
display(pd.DataFrame(list(gradients.items()), columns=["component", "gradient norm"]))
model.zero_grad(set_to_none=True)

The synthetic fixture tests the computation using fixed random weights. The runnable `project.energy_attachment` entry point also accepts native before/after observations and recorded game traces. See [run instructions](README.md#running-the-python-prototype).

Open [arena.ipynb](arena.ipynb) to compare agents and checkpoints in first-to-three series.